#load data


In [ ]:
# Install correct PyTorch + Torchvision for Colab
!pip install -q torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121

# Other required libraries
!pip install -q transformers timm pytorch-lightning scikit-learn pandas opencv-python pillow grad-cam albumentations kaggle tqdm

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.8/7.8 MB 96.5 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 831.6/831.6 kB 54.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 983.2/983.2 kB 55.4 MB/s eta 0:00:00


In [ ]:
# Download APTOS Dataset
import os
import kagglehub

# Download the dataset using kagglehub
path = kagglehub.dataset_download("mariaherrerot/aptos2019")
print("Dataset location:", path)

# Walk through the downloaded directory to find image folders
for root, dirs, files in os.walk(path):
    if any(fname.endswith('.png') for fname in files):
        print("Image folder found:", root)

100%|██████████| 8.01G/8.01G [03:37<00:00, 39.5MB/s]

Extracting files...


Dataset location: /root/.cache/kagglehub/datasets/mariaherrerot/aptos2019/versions/3
Image folder found: /root/.cache/kagglehub/datasets/mariaherrerot/aptos2019/versions/3/val_images/val_images
Image folder found: /root/.cache/kagglehub/datasets/mariaherrerot/aptos2019/versions/3/test_images/test_images
Image folder found: /root/.cache/kagglehub/datasets/mariaherrerot/aptos2019/versions/3/train_images/train_images


In [ ]:
# Import Required Libraries
import random
import math
from pathlib import Path
from PIL import Image
import numpy as np
import pandas as pd
import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as T
from torchvision import models
from transformers import CLIPProcessor, CLIPModel
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, f1_score
from sklearn.metrics import cohen_kappa_score
import matplotlib.pyplot as plt
from tqdm import tqdm

print(' Libraries imported')

 Libraries imported


In [ ]:
# Load and Split Dataset
# Search for train_1.csv within the downloaded dataset path
TRAIN_CSV_NAME = 'train_1.csv'
TRAIN_CSV = None
for root, dirs, files in os.walk(path):
    if TRAIN_CSV_NAME in files:
        TRAIN_CSV = os.path.join(root, TRAIN_CSV_NAME)
        break

if not TRAIN_CSV or not os.path.exists(TRAIN_CSV):
    raise FileNotFoundError(f'{TRAIN_CSV_NAME} not found in the dataset path {path} or its subdirectories.')

# Load the training data into a pandas DataFrame
df = pd.read_csv(TRAIN_CSV)
print('Loaded train_1.csv with', len(df), 'rows')
print('Class distribution:\n', df['diagnosis'].value_counts().sort_index())

# Split the data into training and validation sets
train_df, val_df = train_test_split(df, test_size=0.15, stratify=df['diagnosis'], random_state=42)
train_df = train_df.reset_index(drop=True)
val_df = val_df.reset_index(drop=True)
print('Train / Val sizes:', len(train_df), len(val_df))

# Find ALL image directories in the dataset
IMAGE_DIRS = []
for root, dirs, files in os.walk(path):
    if any(fname.endswith(('.png', '.jpg', '.jpeg')) for fname in files):
        IMAGE_DIRS.append(root)
        print(f"Found image directory: {root}")

if not IMAGE_DIRS:
    raise FileNotFoundError(f'No image directories found in {path}')

print(f"\nTotal image directories found: {len(IMAGE_DIRS)}")

Loaded train_1.csv with 2930 rows
Class distribution:
 diagnosis
0    1434
1     300
2     808
3     154
4     234
Name: count, dtype: int64
Train / Val sizes: 2490 440
Found image directory: /root/.cache/kagglehub/datasets/mariaherrerot/aptos2019/versions/3/val_images/val_images
Found image directory: /root/.cache/kagglehub/datasets/mariaherrerot/aptos2019/versions/3/test_images/test_images
Found image directory: /root/.cache/kagglehub/datasets/mariaherrerot/aptos2019/versions/3/train_images/train_images

Total image directories found: 3


In [ ]:
class AptosDataset(Dataset):
    def __init__(self, df, image_dirs, clip_processor, transform_cnn=None, transform_clip=None):
        """
        Fixed dataset class that searches multiple directories

        Args:
            df: DataFrame with 'id_code' and 'diagnosis' columns
            image_dirs: List of directories to search for images
            clip_processor: CLIP processor for image preprocessing
            transform_cnn: Transforms for CNN model
            transform_clip: Transforms for CLIP model
        """
        self.df = df.reset_index(drop=True)
        self.image_dirs = [Path(d) for d in image_dirs]
        self.clip_processor = clip_processor
        self.transform_cnn = transform_cnn
        self.transform_clip = transform_clip

        # Build image path cache for faster lookup
        self.image_cache = {}
        print(f" Building image cache for {len(self.df)} samples...")
        missing_images = []

        for idx in range(len(self.df)):
            img_id = self.df.iloc[idx]['id_code']
            found = False

            # Search in all image directories
            for img_dir in self.image_dirs:
                for ext in ['png', 'jpg', 'jpeg']:
                    img_path = img_dir / f"{img_id}.{ext}"
                    if img_path.exists():
                        self.image_cache[img_id] = img_path
                        found = True
                        break
                if found:
                    break

            if not found:
                missing_images.append(img_id)

        if missing_images:
            print(f" WARNING: {len(missing_images)} images not found out of {len(self.df)}")
            print(f"First 10 missing: {missing_images[:10]}")
            # Remove missing images from dataframe
            self.df = self.df[~self.df['id_code'].isin(missing_images)].reset_index(drop=True)
            print(f" Continuing with {len(self.df)} valid images")
        else:
            print(f" All {len(self.df)} images found successfully!")

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img_id = row['id_code']
        label = int(row['diagnosis'])

        # Get image path from cache
        if img_id not in self.image_cache:
            raise FileNotFoundError(f'Image for {img_id} not found')

        img_path = self.image_cache[img_id]
        img = Image.open(img_path).convert('RGB')

        # Apply CNN transforms
        if self.transform_cnn is not None:
            img_cnn = self.transform_cnn(img)
        else:
            img_cnn = T.ToTensor()(img)

        # For CLIP, return PIL image
        if self.transform_clip is not None:
            img_clip = self.transform_clip(img)
        else:
            img_clip = img

        return img_cnn, img_clip, label, img_id


def collate_fn(batch, clip_processor):
    """Custom collate function to handle CLIP preprocessing"""
    imgs_cnn = torch.stack([b[0] for b in batch])
    imgs_pil = [b[1] for b in batch]
    labels = torch.tensor([b[2] for b in batch], dtype=torch.long)
    ids = [b[3] for b in batch]

    # Process images for CLIP
    clip_inputs = clip_processor(images=imgs_pil, return_tensors='pt')
    pixel_values = clip_inputs['pixel_values']

    # Return CPU tensors; move to device in the main training loop
    return imgs_cnn, pixel_values, labels, ids

print(' Fixed Dataset class defined')

 Fixed Dataset class defined


# Transformation and BatchLoaders

In [ ]:
# Setup Device and Transforms
import os
# Set the device for training
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device)
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB')


# Define image transformations for the CNN
size = 512
transform_cnn = T.Compose([
    T.Resize((size, size)),
    T.RandomRotation(15),
    T.RandomHorizontalFlip(),
    T.ColorJitter(brightness=0.2, contrast=0.2),
    T.ToTensor(),
    T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

transform_cnn_val = T.Compose([
    T.Resize((size, size)),
    T.ToTensor(),
    T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# Initialize the CLIP processor
clip_model_name = 'openai/clip-vit-base-patch32'
clip_processor = CLIPProcessor.from_pretrained(clip_model_name)

# Create dataset instances for training and validation
# Pass the list of image directories (IMAGE_DIRS) to AptosDataset
train_ds = AptosDataset(train_df, IMAGE_DIRS, clip_processor, transform_cnn=transform_cnn, transform_clip=None)
val_ds = AptosDataset(val_df, IMAGE_DIRS, clip_processor, transform_cnn=transform_cnn_val, transform_clip=None)

# Create DataLoaders for training and validation
batch_size = 64 # Decreased batch size to address OutOfMemoryError
num_workers = os.cpu_count()
print(f"Using {num_workers} workers for DataLoaders")
train_loader = DataLoader(
    train_ds,
    batch_size=batch_size,
    shuffle=True,
    collate_fn=lambda b: collate_fn(b, clip_processor),
    num_workers=num_workers,
    pin_memory=True
)
val_loader = DataLoader(
    val_ds,
    batch_size=batch_size,
    shuffle=False,
    collate_fn=lambda b: collate_fn(b, clip_processor),
    num_workers=num_workers,
    pin_memory=True
)

print('DataLoaders ready - batches per epoch (train, val):', len(train_loader), len(val_loader))

# Test the DataLoader
try:
    test_batch = next(iter(train_loader))
    imgs_cnn, pixel_values, labels, ids = test_batch
    # Move to device here
    imgs_cnn = imgs_cnn.to(device)
    pixel_values = pixel_values.to(device)
    labels = labels.to(device)

    print(f" DataLoader test PASSED!")
    print(f"   - Batch CNN images shape: {imgs_cnn.shape}")
    print(f"   - Batch CLIP pixels shape: {pixel_values.shape}")
    print(f"   - Labels: {labels.tolist()}")
    print(f"   - IDs: {ids[:3]}...")
except Exception as e:
    print(f" DataLoader test FAILED: {e}")
    raise

Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.


Device: cuda
GPU: NVIDIA A100-SXM4-80GB
GPU Memory: 85.17 GB


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


preprocessor_config.json:   0%|          | 0.00/316 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/592 [00:00<?, ?B/s]

...

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/389 [00:00<?, ?B/s]

 Building image cache for 2490 samples...
 All 2490 images found successfully!
 Building image cache for 440 samples...
 All 440 images found successfully!
Using 12 workers for DataLoaders
DataLoaders ready - batches per epoch (train, val): 39 7
 DataLoader test PASSED!
   - Batch CNN images shape: torch.Size([64, 3, 512, 512])
   - Batch CLIP pixels shape: torch.Size([64, 3, 224, 224])
   - Labels: [2, 0, 0, 2, 2, 2, 2, 0, 1, 0, 2, 2, 1, 0, 0, 3, 1, 0, 0, 2, 1, 0, 0, 1, 0, 0, 2, 0, 2, 4, 2, 2, 0, 1, 2, 0, 1, 0, 0, 0, 0, 0, 2, 4, 0, 4, 4, 2, 0, 0, 0, 3, 3, 0, 0, 0, 2, 1, 2, 0, 2, 0, 2, 1]
   - IDs: ['4489d421e5aa', 'a987aa7aac37', '240b25a7debe']...


In [ ]:
# Cell 7: Define Fusion Model
class CLIP_DenseNet_Fusion(nn.Module):
    def __init__(self, clip_model_name='openai/clip-vit-base-patch32', cnn_name='densenet121', num_classes=5, pretrained_cnn=True):
        super().__init__()
        # Load CLIP model
        self.clip = CLIPModel.from_pretrained(clip_model_name)
        try:
            clip_dim = self.clip.visual_projection.out_features
        except:
            clip_dim = self.clip.config.projection_dim if hasattr(self.clip.config, 'projection_dim') else 512

        # Load DenseNet model
        if cnn_name.startswith('densenet'):
            if pretrained_cnn:
                cnn = getattr(models, cnn_name)(weights='DEFAULT')
            else:
                cnn = getattr(models, cnn_name)(weights=None)
            feat_dim = cnn.classifier.in_features
            cnn.classifier = nn.Identity()
            self.cnn = cnn
        else:
            raise NotImplementedError('Only densenet variants implemented')

        # Define the fusion layer dimension
        self.fusion_dim = clip_dim + feat_dim

        # Define the classifier head
        self.classifier = nn.Sequential(
            nn.Linear(self.fusion_dim, 512),
            nn.ReLU(),
            nn.Dropout(0.4),
            nn.Linear(512, 256),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(256, num_classes)
        )

    def forward(self, x_cnn, pixel_values):
        # Get CNN features
        feat_map = self.cnn.features(x_cnn)
        pooled = nn.functional.adaptive_avg_pool2d(feat_map, (1,1)).view(x_cnn.size(0), -1)

        # Get CLIP image features
        clip_outputs = self.clip.get_image_features(pixel_values=pixel_values)

        # Concatenate features
        fused = torch.cat([pooled, clip_outputs], dim=1)

        # Pass through classifier
        out = self.classifier(fused)
        return out, pooled, clip_outputs

# Instantiate the model
num_classes = 5
model = CLIP_DenseNet_Fusion(clip_model_name=clip_model_name, cnn_name='densenet121', num_classes=num_classes).to(device)
print('Model instantiated')
print(f'Total parameters: {sum(p.numel() for p in model.parameters()):,}')
print(f'Trainable parameters: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}')


pytorch_model.bin:   0%|          | 0.00/605M [00:00<?, ?B/s]

Downloading: "https://download.pytorch.org/models/densenet121-a639ec97.pth" to /root/.cache/torch/hub/checkpoints/densenet121-a639ec97.pth


100%|██████████| 30.8M/30.8M [00:00<00:00, 208MB/s]


model.safetensors:   0%|          | 0.00/605M [00:00<?, ?B/s]

Model instantiated
Total parameters: 159,150,726
Trainable parameters: 159,150,726


In [ ]:
def train_epoch(model, loader, optimizer, criterion):
    model.train()
    losses = []
    preds = []
    trues = []

    # Wrap the loader with tqdm for a progress bar
    for x_cnn, pixel_values, labels, ids in tqdm(loader, desc="Training"): # Added tqdm here
        # Move batch to device
        x_cnn = x_cnn.to(device)
        pixel_values = pixel_values.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()

        # Forward pass
        logits, _, _ = model(x_cnn, pixel_values)

        # Calculate loss
        loss = criterion(logits, labels)

        # Backward pass and optimize
        loss.backward()
        optimizer.step()

        losses.append(loss.item())
        preds.extend(torch.argmax(logits, dim=1).cpu().numpy().tolist())
        trues.extend(labels.cpu().numpy().tolist())

    return np.mean(losses), accuracy_score(trues, preds), f1_score(trues, preds, average='macro')


def validate_epoch(model, loader, criterion):
    model.eval()
    losses = []
    preds = []
    trues = []

    with torch.no_grad():
        # Wrap the loader with tqdm for a progress bar
        for x_cnn, pixel_values, labels, ids in tqdm(loader, desc="Validation"): # Added tqdm here
            # Move batch to device
            x_cnn = x_cnn.to(device)
            pixel_values = pixel_values.to(device)
            labels = labels.to(device)

            # Forward pass
            logits, _, _ = model(x_cnn, pixel_values)

            # Calculate loss
            loss = criterion(logits, labels)
            losses.append(loss.item())
            preds.extend(torch.argmax(logits, dim=1).cpu().numpy().tolist())
            trues.extend(labels.cpu().numpy().tolist())

    return np.mean(losses), accuracy_score(trues, preds), f1_score(trues, preds, average='macro'), trues, preds

# Train

In [ ]:
# Cell 9: Linear Probe Training
# Linear-probe training: freeze CLIP & CNN, train only classifier
for p in model.clip.parameters():
    p.requires_grad = False
for p in model.cnn.parameters():
    p.requires_grad = False
for p in model.classifier.parameters():
    p.requires_grad = True

# Define optimizer and loss function for linear probing
optimizer = torch.optim.AdamW(model.classifier.parameters(), lr=1e-3, weight_decay=1e-4)
criterion = nn.CrossEntropyLoss()

# Train the classifier head
n_epochs = 20
best_val_f1 = 0.0
history = {'train_loss': [], 'train_acc': [], 'train_f1': [], 'val_loss': [], 'val_acc': [], 'val_f1': []}

print("="*60)
print("STAGE 1: LINEAR PROBE TRAINING")
print("="*60)

for epoch in range(n_epochs):
    print(f'\n=== Epoch {epoch+1}/{n_epochs} (Linear probe) ===')
    train_loss, train_acc, train_f1 = train_epoch(model, train_loader, optimizer, criterion)
    val_loss, val_acc, val_f1, val_trues, val_preds = validate_epoch(model, val_loader, criterion)

    # Store history
    history['train_loss'].append(train_loss)
    history['train_acc'].append(train_acc)
    history['train_f1'].append(train_f1)
    history['val_loss'].append(val_loss)
    history['val_acc'].append(val_acc)
    history['val_f1'].append(val_f1)

    print(f'Train | Loss: {train_loss:.4f} | Acc: {train_acc:.4f} | F1: {train_f1:.4f}')
    print(f'Val   | Loss: {val_loss:.4f} | Acc: {val_acc:.4f} | F1: {val_f1:.4f}')

    # Save the best model based on validation F1 score
    if val_f1 > best_val_f1:
        best_val_f1 = val_f1
        torch.save(model.state_dict(), 'best_linear_probe.pth')
        print(f' New best model saved! Val F1: {best_val_f1:.4f}')

print(f'\n{"="*60}')
print(f'Linear probe finished | Best Val F1: {best_val_f1:.4f}')
print("="*60)

STAGE 1: LINEAR PROBE TRAINING

=== Epoch 1/20 (Linear probe) ===



Validation: 100%|██████████| 7/7 [00:16<00:00,  2.33s/it]


Train | Loss: 0.8998 | Acc: 0.6735 | F1: 0.3137
Val   | Loss: 0.6810 | Acc: 0.7409 | F1: 0.3807
 New best model saved! Val F1: 0.3807

=== Epoch 2/20 (Linear probe) ===


Validation: 100%|██████████| 7/7 [00:16<00:00,  2.34s/it]


Train | Loss: 0.6833 | Acc: 0.7442 | F1: 0.4089
Val   | Loss: 0.5763 | Acc: 0.7773 | F1: 0.4545
 New best model saved! Val F1: 0.4545

=== Epoch 3/20 (Linear probe) ===


...

Train | Loss: 0.4421 | Acc: 0.8309 | F1: 0.6681
Val   | Loss: 0.4746 | Acc: 0.8273 | F1: 0.6446

=== Epoch 19/20 (Linear probe) ===


Validation: 100%|██████████| 7/7 [00:17<00:00,  2.47s/it]


Train | Loss: 0.4217 | Acc: 0.8438 | F1: 0.6959
Val   | Loss: 0.4817 | Acc: 0.8341 | F1: 0.6460

=== Epoch 20/20 (Linear probe) ===


Validation: 100%|██████████| 7/7 [00:17<00:00,  2.48s/it]

Train | Loss: 0.4215 | Acc: 0.8414 | F1: 0.6903
Val   | Loss: 0.4869 | Acc: 0.8227 | F1: 0.6275

Linear probe finished | Best Val F1: 0.6748


In [ ]:
import torch
import gc

print("Clearing CUDA cache...")
gc.collect()
torch.cuda.empty_cache()
print("CUDA cache cleared.")

Clearing CUDA cache...
CUDA cache cleared.


In [ ]:
# Fine-tuning: unfreeze parts of the CLIP & CNN
for p in model.clip.parameters():
    p.requires_grad = True
for name, p in model.cnn.named_parameters():
    if 'denseblock1' in name or 'norm0' in name:
        p.requires_grad = False
    else:
        p.requires_grad = True

# Define optimizer for fine-tuning (only for trainable parameters)
params = [p for p in model.parameters() if p.requires_grad]
optimizer = torch.optim.AdamW(params, lr=1e-5, weight_decay=1e-6)

# Fine-tune the entire model
best_val_f1 = 0.0
for epoch in range(n_epochs):
    print(f'=== Epoch {epoch+1}/{n_epochs} (Fine-tune) ===')
    train_loss, train_acc, train_f1 = train_epoch(model, train_loader, optimizer, criterion)
    val_loss, val_acc, val_f1, val_trues, val_preds = validate_epoch(model, val_loader, criterion)
    print(f'Train loss {train_loss:.4f} acc {train_acc:.4f} f1 {train_f1:.4f}')
    print(f'Val   loss {val_loss:.4f} acc {val_acc:.4f} f1 {val_f1:.4f}')
    # Save the best model based on validation F1 score
    if val_f1 > best_val_f1:
        best_val_f1 = val_f1
        torch.save(model.state_dict(), 'best_finetune.pth')

print('Fine-tuning finished')

=== Epoch 1/20 (Fine-tune) ===


Validation: 100%|██████████| 7/7 [00:17<00:00,  2.48s/it]


Train loss 0.5252 acc 0.8169 f1 0.6587
Val   loss 0.5513 acc 0.8159 f1 0.6759
=== Epoch 2/20 (Fine-tune) ===


Validation: 100%|██████████| 7/7 [00:16<00:00,  2.35s/it]


Train loss 0.4191 acc 0.8390 f1 0.6823
Val   loss 0.4686 acc 0.8318 f1 0.6700
=== Epoch 3/20 (Fine-tune) ===


...

Train loss 0.0522 acc 0.9835 f1 0.9696
Val   loss 0.7855 acc 0.8386 f1 0.6850
=== Epoch 19/20 (Fine-tune) ===


Validation: 100%|██████████| 7/7 [00:17<00:00,  2.46s/it]


Train loss 0.0489 acc 0.9831 f1 0.9678
Val   loss 0.7344 acc 0.8364 f1 0.7096
=== Epoch 20/20 (Fine-tune) ===


Validation: 100%|██████████| 7/7 [00:15<00:00,  2.25s/it]


Train loss 0.0444 acc 0.9843 f1 0.9695
Val   loss 1.0165 acc 0.8227 f1 0.6489
Fine-tuning finished


In [ ]:
# Load the best fine-tuned model
model.load_state_dict(torch.load('best_finetune.pth', map_location=device))
model.eval()
# Evaluate the model on the validation set
_, _, _, val_trues, val_preds = validate_epoch(model, val_loader, criterion)
# Print evaluation metrics
print('Classification Report:')
print(classification_report(val_trues, val_preds, digits=4))
print('Confusion Matrix:')
print(confusion_matrix( val_trues, val_preds))
print('Quadratic weighted kappa:', cohen_kappa_score(val_trues, val_preds, weights='quadratic'))

Validation: 100%|██████████| 7/7 [00:17<00:00,  2.48s/it]

Classification Report:
              precision    recall  f1-score   support

           0     0.9682    0.9861    0.9771       216
           1     0.6667    0.5778    0.6190        45
           2     0.7429    0.8595    0.7969       121
           3     0.6667    0.4348    0.5263        23
           4     0.8846    0.6571    0.7541        35

    accuracy                         0.8545       440
   macro avg     0.7858    0.7031    0.7347       440
weighted avg     0.8530    0.8545    0.8496       440

Confusion Matrix:
[[213   1   2   0   0]
 [  7  26  12   0   0]
 [  0  12 104   3   2]
 [  0   0  12  10   1]
 [  0   0  10   2  23]]
Quadratic weighted kappa: 0.9237888572250255
